# Potential Talents: TF–IDF, Word2Vec, and Sentence-BERT Ranking

**Objective:** rank candidates for an aspiring or job-seeking human resources role, then update Word2Vec and Sentence-BERT rankings using a starred candidate.

**Duplicate policy:** rank each unique cleaned job description once. Keep the lowest candidate ID as its representative and retain all associated IDs for reference. Matching titles do not prove that candidates are the same person.


## Import packages

Import only the tools used below. Keep standard package aliases and descriptive project variable names.


In [48]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import gensim.downloader as gensim_downloader
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer


## Load the candidate data

Use your existing Windows path or place the CSV beside the notebook. Update the path if needed.

In [49]:
# Start with your existing CSV path.
CANDIDATE_DATA_PATH = Path("../data/Potential_Talents_Aspiring_human_resources.csv")

# If unavailable, look for the documented filenames beside the notebook.
if not CANDIDATE_DATA_PATH.exists():
    for csv_filename in [
        'Potential_Talents_Aspiring_human_resources.csv',
        'potential-talents - Aspiring human resources - seeking human resources.csv'
    ]:
        if Path(csv_filename).exists():
            CANDIDATE_DATA_PATH = Path(csv_filename)
            break

if not CANDIDATE_DATA_PATH.exists():
    raise FileNotFoundError(f'Upload the CSV or update CANDIDATE_DATA_PATH: {CANDIDATE_DATA_PATH}')

original_candidate_data = pd.read_csv(CANDIDATE_DATA_PATH)
display(original_candidate_data.head())

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


## Inspect the structure 

Inspect the dataset before configuring the ranking methods.


In [50]:
print(f'Rows: {original_candidate_data.shape[0]:,} | Columns: {original_candidate_data.shape[1]}')
original_candidate_data.info()



Rows: 104 | Columns: 5
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          104 non-null    int64  
 1   job_title   104 non-null    object 
 2   location    104 non-null    object 
 3   connection  104 non-null    object 
 4   fit         0 non-null      float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.2+ KB


### Check missing values
Determine whether fields require cleaning and, importantly, whether the desired target contains any labels.

In [51]:
missing_value_summary = pd.DataFrame({
    'missing_count':original_candidate_data.isna().sum(),
    'missing_percent':original_candidate_data.isna().mean().mul(100).round(1)
})
display(missing_value_summary)

,missing_count,missing_percent
id,0,0.0
job_title,0,0.0
location,0,0.0
connection,0,0.0
fit,104,100.0


#### Interpretation for the supplied CSV: 
- The fit column is entirely missing, all 104 fit values are missing;
- Candidate information is complete across all other input columns.
- So initial candidate ranking requires similarity-based scoring rather than supervised learning.

## Standardize columns and validate IDs

Standardize the schema after inspecting it. Validate the unique identifier before linking rankings and feedback.

In [52]:
original_candidate_data.columns = original_candidate_data.columns.str.strip().str.lower()
if 'connections' in original_candidate_data.columns and 'connection' not in original_candidate_data.columns:
    original_candidate_data = original_candidate_data.rename(columns={'connections':'connection'})

required_column_names = {'id','job_title','location','connection'}
missing_column_names = required_column_names - set(original_candidate_data.columns)
if missing_column_names:
    raise ValueError(f'Missing required columns: {missing_column_names}')
if original_candidate_data.empty:
    raise ValueError('The CSV has no candidates.')

original_candidate_data['id'] = pd.to_numeric(original_candidate_data['id'], errors='raise')
if original_candidate_data['id'].isna().any() or (original_candidate_data['id'] % 1 != 0).any():
    raise ValueError('Candidate IDs must be nonmissing whole numbers.')
original_candidate_data['id'] = original_candidate_data['id'].astype(int)
if original_candidate_data['id'].duplicated().any():
    raise ValueError('Resolve duplicate candidate IDs before ranking.')
print('Passed: required columns are present and every candidate ID is unique.')


Passed: required columns are present and every candidate ID is unique.


## Analyze repeated profiles and representative titles

Inspect the original records first. After cleaning, keep one representative per cleaned title for ranking. Preserve the original records and the mapping of all IDs to their representative.

In [53]:
profile_comparison_columns = ['job_title','location','connection']
candidate_data_summary = pd.Series({
    'candidate_records':len(original_candidate_data),
    'unique_ids':original_candidate_data['id'].nunique(),
    'unique_original_titles':original_candidate_data['job_title'].nunique(),
    'repeated_profile_rows_after_first':original_candidate_data.duplicated(profile_comparison_columns).sum()
}, name='value')
display(candidate_data_summary.to_frame())

#Creates a list of candidate IDs to inspect
example_candidate_ids = [3,6,8,12,28,75,80,98,100]
# Temporarily allows pandas to display up to 130 characters per cell, making longer job titles easier to read.
with pd.option_context('display.max_colwidth', 130):
    display(original_candidate_data.loc[
        original_candidate_data['id'].isin(example_candidate_ids), ['id','job_title']
    ])


,value
candidate_records,104
unique_ids,104
unique_original_titles,52
repeated_profile_rows_after_first,51


,id,job_title
2,3,Aspiring Human Resources Professional
5,6,Aspiring Human Resources Specialist
7,8,HR Senior Specialist
11,12,"SVP, CHRO, Marketing & Communications, CSR Officer | ENGIE | Houston | The Woodlands | Energy | GPHR | SPHR"
27,28,Seeking Human Resources Opportunities
74,75,"Nortia Staffing is seeking Human Resources, Payroll & Administrative Professionals!! (408) 709-2621"
79,80,Junior MES Engineer| Information Systems
97,98,Student
99,100,Aspiring Human Resources Manager | Graduating May 2020 | Seeking an Entry-Level Human Resources Position in St. Louis


**Interpretation:** 
- the supplied data has 104 unique IDs, 52 original titles, and 51 repeated profile rows after their first occurrence. 
- Candidate 75 appears to be an employer advertisement; candidate 98 has a sparse title. Flag these descriptions for review rather than automatically rejecting them.

**Decisions:** expand HR abbreviations
- preserve intent and seniority
- remove contact details,
- exclude location and connections from scoring, and compare TF–IDF with Word2Vec.
- Rank identical cleaned titles only once and retain their associated IDs for reference. Group identical titles if independent evaluation is added later.


##  Configure the search

These are the only ranking settings needed for this version. Change the search phrases when starting another role and clear the old starred IDs.


In [54]:
ROLE_SEARCH_PHRASES = ['Aspiring human resources', 'seeking human resources']
FEEDBACK_WEIGHT = 0.30  # Starting choice; validate using independent reviewer labels.
DISPLAY_CANDIDATE_COUNT = 15
RESULTS_DIRECTORY = Path('candidate_ranking_outputs')
RESULTS_DIRECTORY.mkdir(exist_ok=True)

SENTENCE_BERT_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'


##  Clean the titles

Keep the original data unchanged. Use one normalization function for both candidate titles and search phrases.


In [55]:
def normalize_job_title(job_title):
    title_text = '' if pd.isna(job_title) else str(job_title)
    
    # Remove email addresses.
    title_text = re.sub(r'\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b', ' ', title_text)
    title_text = re.sub(r'(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)', ' ', title_text)
    title_text = title_text.lower()
    title_text = re.sub(r'\bh\.?\s*r\.?\b', 'human resources', title_text)
    title_text = re.sub(r'\bhuman resource\b', 'human resources', title_text)
    title_text = re.sub(r'[^a-z0-9\s]', ' ', title_text)
    return re.sub(r'\s+', ' ', title_text).strip()

prepared_candidate_data = original_candidate_data.copy().reset_index(drop=True)
# Remove contact details from the readable title as well as the modeling text.
prepared_candidate_data['job_title'] = (
    prepared_candidate_data['job_title'].fillna('').astype(str)
    .str.replace(r'\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b', ' ', regex=True)
    .str.replace(r'(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True).str.strip()
)
prepared_candidate_data['clean_job_title'] = prepared_candidate_data['job_title'].map(normalize_job_title)
normalized_search_phrases = [normalize_job_title(search_phrase) for search_phrase in ROLE_SEARCH_PHRASES]
display(prepared_candidate_data[['id','job_title','clean_job_title']].head(10))


,id,job_title,clean_job_title
0,1,2019 C.T. Bauer College of Business Graduate (...,2019 c t bauer college of business graduate ma...
1,2,Native English Teacher at EPIK (English Progra...,native english teacher at epik english program...
2,3,Aspiring Human Resources Professional,aspiring human resources professional
3,4,People Development Coordinator at Ryan,people development coordinator at ryan
4,5,Advisory Board Member at Celal Bayar University,advisory board member at celal bayar university
5,6,Aspiring Human Resources Specialist,aspiring human resources specialist
6,7,Student at Humber College and Aspiring Human R...,student at humber college and aspiring human r...
7,8,HR Senior Specialist,human resources senior specialist
8,9,Student at Humber College and Aspiring Human R...,student at humber college and aspiring human r...
9,10,Seeking Human Resources HRIS and Generalist Po...,seeking human resources hris and generalist po...


## Remove duplicate job descriptions

Keep one row per `clean_job_title`, using the lowest ID as the representative. All ranking steps below use this same deduplicated table. Different original spellings that normalize to the same text are grouped together.

Original rows are retained in `all_candidate_records`. `candidate_description_mapping` links every original ID to its representative. The representative's location and connection details belong to that record; they are not merged qualifications and do not affect scoring.

In [56]:
# Preserve every original candidate record before removing repeated descriptions.
all_candidate_records = prepared_candidate_data.copy()

# Sort by ID so the same representative is chosen each time.
all_candidate_records = all_candidate_records.sort_values('id').reset_index(drop=True)

# Collect every ID associated with each cleaned description.
description_summary = all_candidate_records.groupby(
    'clean_job_title', sort=False
).agg(
    representative_id=('id', 'first'),
    associated_candidate_ids=('id', list),
    description_record_count=('id', 'size')
).reset_index()

# Preserve a mapping from every original candidate ID to its representative.
candidate_description_mapping = all_candidate_records[
    ['id', 'clean_job_title']
].merge(description_summary, on='clean_job_title', validate='many_to_one')

# Keep the first (lowest-ID) row for each cleaned description.
prepared_candidate_data = all_candidate_records.drop_duplicates(
    subset='clean_job_title', keep='first'
).copy()

# Attach the original IDs and record count to each representative.
prepared_candidate_data = prepared_candidate_data.merge(
    description_summary.drop(columns='representative_id'),
    on='clean_job_title', validate='one_to_one'
)

# Reset the index so DataFrame rows align with vector matrix positions.
prepared_candidate_data = prepared_candidate_data.reset_index(drop=True)

# Confirm there is only one row per cleaned description.
assert prepared_candidate_data['clean_job_title'].is_unique
assert len(candidate_description_mapping) == len(all_candidate_records)

# Report the actual counts instead of assuming a fixed dataset size.
print('Original candidate records:', len(all_candidate_records))
print('Unique cleaned descriptions:', len(prepared_candidate_data))
print('Repeated descriptions removed:', len(all_candidate_records) - len(prepared_candidate_data))

# Inspect descriptions that originally appeared more than once.
display(description_summary.loc[
    description_summary['description_record_count'] > 1,
    ['representative_id', 'clean_job_title', 'associated_candidate_ids', 'description_record_count']
])

# Preview the unique descriptions used by all three ranking methods.
display(prepared_candidate_data[
    ['id', 'job_title', 'clean_job_title', 'associated_candidate_ids']
].head(10))

Original candidate records: 104
Unique cleaned descriptions: 52
Repeated descriptions removed: 52


,representative_id,clean_job_title,associated_candidate_ids,description_record_count
0,1,2019 c t bauer college of business graduate ma...,"[1, 14, 15, 19, 31, 44, 57]",7
1,2,native english teacher at epik english program...,"[2, 16, 20, 32, 45]",5
2,3,aspiring human resources professional,"[3, 17, 21, 33, 46, 58, 97]",7
3,4,people development coordinator at ryan,"[4, 18, 22, 34, 47, 59]",6
4,5,advisory board member at celal bayar university,"[5, 23, 35, 48]",4
5,6,aspiring human resources specialist,"[6, 24, 36, 49, 60]",5
6,7,student at humber college and aspiring human r...,"[7, 9, 25, 37, 39, 50, 52]",7
7,8,human resources senior specialist,"[8, 26, 38, 51, 61]",5
8,10,seeking human resources hris and generalist po...,"[10, 40, 53, 62]",4
9,11,student at chapman university,"[11, 41, 54, 63]",4


,id,job_title,clean_job_title,associated_candidate_ids
0,1,2019 C.T. Bauer College of Business Graduate (...,2019 c t bauer college of business graduate ma...,"[1, 14, 15, 19, 31, 44, 57]"
1,2,Native English Teacher at EPIK (English Progra...,native english teacher at epik english program...,"[2, 16, 20, 32, 45]"
2,3,Aspiring Human Resources Professional,aspiring human resources professional,"[3, 17, 21, 33, 46, 58, 97]"
3,4,People Development Coordinator at Ryan,people development coordinator at ryan,"[4, 18, 22, 34, 47, 59]"
4,5,Advisory Board Member at Celal Bayar University,advisory board member at celal bayar university,"[5, 23, 35, 48]"
5,6,Aspiring Human Resources Specialist,aspiring human resources specialist,"[6, 24, 36, 49, 60]"
6,7,Student at Humber College and Aspiring Human R...,student at humber college and aspiring human r...,"[7, 9, 25, 37, 39, 50, 52]"
7,8,HR Senior Specialist,human resources senior specialist,"[8, 26, 38, 51, 61]"
8,10,Seeking Human Resources HRIS and Generalist Po...,seeking human resources hris and generalist po...,"[10, 40, 53, 62]"
9,11,Student at Chapman University,student at chapman university,"[11, 41, 54, 63]"


## Flag titles needing human review

Flags remain separate from relevance scores. The working table now contains one row per cleaned title. Original candidate IDs remain available in `all_candidate_records` and `candidate_description_mapping`.

In [57]:
# repeated_title indicates whether this description appeared more than once in the original data.
prepared_candidate_data['possible_advertisement'] = prepared_candidate_data['clean_job_title'].str.contains(
    r'\b(?:staffing|recruitment agency|we are hiring|hiring for|seeking .* professionals)\b', regex=True
)
prepared_candidate_data['sparse_title'] = prepared_candidate_data['clean_job_title'].str.split().str.len().lt(3)
prepared_candidate_data['repeated_title'] = prepared_candidate_data['description_record_count'].gt(1)

#Select candidates flagged as possible advertisements OR as having sparse titles.
display(prepared_candidate_data.loc[
    prepared_candidate_data['possible_advertisement'] | prepared_candidate_data['sparse_title'],
    ['id','job_title','possible_advertisement','sparse_title']
])


,id,job_title,possible_advertisement,sparse_title
15,67,"Human Resources, Staffing and Recruiting Profe...",True,False
23,75,"Nortia Staffing is seeking Human Resources, Pa...",True,False
45,98,Student,False,True


## Build the TF–IDF baseline

Fit on unique titles and rank each description once. Keep the candidate representation sparse. Average the search phrases and calculate cosine similarity.

In [58]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, sublinear_tf=True)
tfidf_vectorizer.fit(prepared_candidate_data['clean_job_title'].drop_duplicates())
candidate_tfidf_vectors = tfidf_vectorizer.transform(prepared_candidate_data['clean_job_title'])
tfidf_search_vectors = tfidf_vectorizer.transform(normalized_search_phrases)
tfidf_role_vector = np.asarray(tfidf_search_vectors.mean(axis=0))
if np.linalg.norm(tfidf_role_vector) == 0:
    raise ValueError('No search words match the TF–IDF vocabulary. Revise the search phrases.')

tfidf_similarity_scores = cosine_similarity(candidate_tfidf_vectors, tfidf_role_vector).ravel()
tfidf_ranking = prepared_candidate_data[['id','job_title']].copy()
tfidf_ranking['tfidf_similarity'] = tfidf_similarity_scores
tfidf_ranking = tfidf_ranking.sort_values(
    ['tfidf_similarity','id'], ascending=[False,True]
).reset_index(drop=True)
tfidf_ranking.insert(0,'tfidf_rank',np.arange(1,len(tfidf_ranking)+1))
display(tfidf_ranking.head(DISPLAY_CANDIDATE_COUNT))


,tfidf_rank,id,job_title,tfidf_similarity
0,1,3,Aspiring Human Resources Professional,0.586905
1,2,28,Seeking Human Resources Opportunities,0.544027
2,3,99,Seeking Human Resources Position,0.544027
3,4,6,Aspiring Human Resources Specialist,0.514304
4,5,73,"Aspiring Human Resources Manager, seeking inte...",0.451356
5,6,74,Human Resources Professional,0.364647
6,7,27,Aspiring Human Resources Management student se...,0.343073
7,8,10,Seeking Human Resources HRIS and Generalist Po...,0.318087
8,9,72,Business Management Major and Aspiring Human R...,0.293029
9,10,66,Experienced Retail Manager and aspiring Human ...,0.284481


**Interpretation:** TF–IDF ranks vocabulary overlap. Each cleaned description appears once. Ties between different descriptions use ascending representative ID for reproducibility; lower IDs do not indicate stronger candidates.

## Load pretrained Word2Vec

Load pretrained Word2Vec vectors. Download once, then reuse the saved model.


In [59]:
# Returns KeyedVectors: access words with pretrained_word_vectors[word], without .wv.
pretrained_word_vectors = gensim_downloader.load('word2vec-google-news-300')
print('Word vector size:', pretrained_word_vectors.vector_size)


Word vector size: 300


##  Convert each title into a Word2Vec vector

- Averaging creates one fixed-length vector from a title’s known words.
- Unknown words are skipped.
- Empty or entirely unknown titles receive a zero vector and are flagged for review.
- The function reads the dimension from the model.


In [60]:
# Define the function and collect word vectors.
def average_title_vector(title_text, word_vectors):
    # Create a list of vectors for words the model knows.
    known_word_vectors = [
        word_vectors[word]             # Get this word's vector.
        for word in title_text.split() # Split the title and loop through its words.
        if word in word_vectors        # Skip words missing from the model.
    ]
    # Check whether no known word vectors were found.
    if not known_word_vectors:
        return np.zeros(word_vectors.vector_size, dtype=np.float32)
    return np.mean(known_word_vectors, axis=0)


In [61]:
# Start a list to store the title vectors.
candidate_title_vectors = []

# Loop through every cleaned job title.
for job_title in prepared_candidate_data['clean_job_title']:

    # Create one vector for this title.
    title_vector = average_title_vector(job_title, pretrained_word_vectors)

    # Add the vector to the list.
    candidate_title_vectors.append(title_vector)

# Stack the vectors into a matrix: one row per candidate.
candidate_word2vec_vectors = np.vstack(candidate_title_vectors)

candidate_word2vec_vectors


array([[-0.08377779,  0.07017165, -0.01047457, ..., -0.03605065,
        -0.01220997,  0.10687725],
       [ 0.01379395,  0.0234375 ,  0.06796265, ...,  0.00350952,
        -0.00569916,  0.00704956],
       [-0.09277344,  0.03723145,  0.08280945, ..., -0.04348755,
        -0.08966064,  0.03756714],
       ...,
       [ 0.13376465, -0.04760742, -0.05231934, ..., -0.01005859,
         0.08846893,  0.0173584 ],
       [ 0.03030396,  0.05655416,  0.06352743, ...,  0.04887899,
        -0.05454509, -0.05508423],
       [-0.06942139, -0.02642822,  0.04277649, ..., -0.00730591,
         0.06840821,  0.10683594]], shape=(52, 300), dtype=float32)

In [62]:
# Check which candidates have usable vectors
# Calculate the length of each candidate's vector.
candidate_vector_lengths = np.linalg.norm(candidate_word2vec_vectors, axis=1)

# Mark nonzero vectors as True and zero vectors as False.
prepared_candidate_data['has_word2vec_vector'] = candidate_vector_lengths > 0

In [63]:
# Normalize vectors
# Rescale each nonzero vector to length 1; zero vectors remain zero.
candidate_word2vec_vectors = normalize(candidate_word2vec_vectors)

In [64]:
# Display the results
# Display the number of candidates and vector dimensions.
print('Candidate vector matrix:', candidate_word2vec_vectors.shape)

# Identify candidates whose vectors contain only zeros.
candidates_without_vectors = ~prepared_candidate_data['has_word2vec_vector']

# Count these candidates and convert the count into a whole number.
missing_vector_count = int(candidates_without_vectors.sum())

# Display the count.
print('Candidates without known words:', missing_vector_count)

Candidate vector matrix: (52, 300)
Candidates without known words: 0


**Interpretation:**
- The matrix has one row per unique cleaned description and 300 columns for Google News Word2Vec. The printed shape gives the actual count.
- Averaging discards word order.
- Word2Vec has one fixed representation per known word, rather than a context-dependent sentence representation.

##  Convert search phrases and rank with Word2Vec

- Represent search phrases with the same pretrained model.
- Print vocabulary gaps and stop if the query has no usable words.
-  `fit = (cosine + 1) / 2` is a bounded relevance score, not a probability.


In [65]:
# Find search words missing from the model
# Start an empty set to collect missing words without duplicates.
unknown_search_words = set()

# Loop through each cleaned search phrase.
for search_phrase in normalized_search_phrases:

    # Split the phrase into words and check each word.
    for word in search_phrase.split():

        # Check whether the model does not contain this word.
        if word not in pretrained_word_vectors:

            # Add the missing word to the set.
            unknown_search_words.add(word)

# Convert the set into an alphabetically sorted list.
unknown_search_words = sorted(unknown_search_words)

# Display words that cannot contribute to the search vectors.
print('Search words missing from the model:', unknown_search_words)



Search words missing from the model: []


In [66]:
unknown_search_words = sorted({
    word for search_phrase in normalized_search_phrases
    for word in search_phrase.split() if word not in pretrained_word_vectors
})
print('Search words missing from the model:', unknown_search_words)

word2vec_search_vectors = np.vstack([
    average_title_vector(search_phrase, pretrained_word_vectors)
    for search_phrase in normalized_search_phrases
])
known_search_vectors = word2vec_search_vectors[np.linalg.norm(word2vec_search_vectors, axis=1) > 0]
if len(known_search_vectors) == 0:
    raise ValueError('No search words are in the Word2Vec vocabulary. Revise the search phrases.')
original_role_vector = normalize(known_search_vectors.mean(axis=0).reshape(1,-1))
if np.linalg.norm(original_role_vector) == 0:
    raise ValueError('The averaged search vector is zero. Revise the search phrases.')

word2vec_similarity_scores = np.clip(
    cosine_similarity(candidate_word2vec_vectors, original_role_vector).ravel(), -1, 1
)
word2vec_ranking = prepared_candidate_data.copy()
word2vec_ranking['word2vec_similarity'] = word2vec_similarity_scores
word2vec_ranking['fit'] = (word2vec_similarity_scores+1)/2
word2vec_ranking = word2vec_ranking.sort_values(
    ['word2vec_similarity','id'], ascending=[False,True]
).reset_index(drop=True)
word2vec_ranking.insert(0,'word2vec_rank',np.arange(1,len(word2vec_ranking)+1))
display(word2vec_ranking[
    ['word2vec_rank','id','job_title','word2vec_similarity','fit']
].head(DISPLAY_CANDIDATE_COUNT))


Search words missing from the model: []


,word2vec_rank,id,job_title,word2vec_similarity,fit
0,1,73,"Aspiring Human Resources Manager, seeking inte...",0.900601,0.950300
1,2,74,Human Resources Professional,0.877760,0.938880
2,3,6,Aspiring Human Resources Specialist,0.872995,0.936497
3,4,3,Aspiring Human Resources Professional,0.871868,0.935934
4,5,28,Seeking Human Resources Opportunities,0.870136,0.935068
5,6,99,Seeking Human Resources Position,0.869522,0.934761
6,7,10,Seeking Human Resources HRIS and Generalist Po...,0.793605,0.896803
7,8,101,Human Resources Generalist at Loparex,0.783363,0.891682
8,9,88,Human Resources Management Major,0.777377,0.888688
9,10,100,Aspiring Human Resources Manager | Graduating ...,0.774377,0.887189


In [67]:
# Create a vector for each search phrase
# Start a list to store the search phrase vectors.
search_phrase_vectors = []

# Loop through each cleaned search phrase.
for search_phrase in normalized_search_phrases:

    # Average the known word vectors in this phrase.
    search_vector = average_title_vector(search_phrase, pretrained_word_vectors)

    # Add the phrase vector to the list.
    search_phrase_vectors.append(search_vector)

# Stack the phrase vectors into a matrix: one row per phrase.
word2vec_search_vectors = np.vstack(search_phrase_vectors)


In [68]:
#Keep usable search vectors
# Average the usable phrase vectors into one vector.
average_search_vector = known_search_vectors.mean(axis=0)

# Reshape the vector into one row for similarity calculations.
original_role_vector = average_search_vector.reshape(1, -1)

# Rescale the role vector to length 1.
original_role_vector = normalize(original_role_vector)

# Check whether averaging produced a zero vector.
if np.linalg.norm(original_role_vector) == 0:

    # Stop because a zero vector cannot represent the search meaningfully.
    raise ValueError(
        'The averaged search vector is zero. Revise the search phrases.'
    )

In [69]:
#Calculate candidate similarity scores
# Combine the search phrases into one role vector
# Compare every candidate vector with the role vector.
candidate_role_similarities = cosine_similarity(
    candidate_word2vec_vectors,
    original_role_vector
)

# Convert the one-column result into a one-dimensional array.
word2vec_similarity_scores = candidate_role_similarities.ravel()

# Keep scores within the valid cosine range, correcting tiny rounding errors.
word2vec_similarity_scores = np.clip(word2vec_similarity_scores, -1, 1)

In [70]:
#Create the results table
# Copy candidate information into a separate ranking table.
word2vec_ranking = prepared_candidate_data.copy()

# Add each candidate's cosine similarity score.
word2vec_ranking['word2vec_similarity'] = word2vec_similarity_scores

# Convert similarity from [-1, 1] into a relevance score in [0, 1].
# This score is not a probability of suitability.
word2vec_ranking['fit'] = (word2vec_similarity_scores + 1) / 2

In [71]:
#Sort candidates and assign ranks
# Sort by highest similarity first; break ties using the lowest ID.
word2vec_ranking = word2vec_ranking.sort_values(
    ['word2vec_similarity', 'id'],
    ascending=[False, True]
)

# Replace the old row index with consecutive numbers starting from zero.
word2vec_ranking = word2vec_ranking.reset_index(drop=True)

# Create rank numbers starting from 1.
candidate_rank_numbers = np.arange(1, len(word2vec_ranking) + 1)

# Insert the rank numbers as the first column.
word2vec_ranking.insert(0, 'word2vec_rank', candidate_rank_numbers)

In [72]:
# Convert the search phrase into a Word2Vec vector.
search_vector = average_title_vector(
    normalize_job_title("Human Resources"),
    pretrained_word_vectors
)

# Compare the search vector with all candidate vectors.
similarity_scores = cosine_similarity(
    candidate_word2vec_vectors,
    search_vector.reshape(1, -1)
).ravel()
similarity_scores

array([0.39384943, 0.15993139, 0.73136264, 0.31142488, 0.19764285,
       0.78298587, 0.63345134, 0.7888726 , 0.718514  , 0.16220255,
       0.38609913, 0.659868  , 0.58162445, 0.7809049 , 0.6122817 ,
       0.72724116, 0.84182346, 0.6368545 , 0.5048857 , 0.7489488 ,
       0.67028844, 0.7883121 , 0.89783263, 0.6779456 , 0.5461458 ,
       0.6592259 , 0.8116733 , 0.581449  , 0.29274505, 0.6972823 ,
       0.56651175, 0.6258099 , 0.6506978 , 0.19842649, 0.392766  ,
       0.24865603, 0.83992064, 0.7409649 , 0.3067266 , 0.20933358,
       0.3383443 , 0.25725973, 0.6100577 , 0.22236055, 0.29765671,
       0.12627442, 0.7829242 , 0.66711736, 0.83857197, 0.31324038,
       0.13925175, 0.27245125], dtype=float32)

In [73]:
# Convert the search phrase into a vector.
search_vector = average_title_vector(
    normalize_job_title("Human Resources"),
    pretrained_word_vectors
)

# Convert a single candidate title into a vector.
title_vector = average_title_vector(
    normalize_job_title("Aspiring Human Resources Professional"),
    pretrained_word_vectors
)

# Calculate similarity between the search and this title.
similarity_score = cosine_similarity(
    search_vector.reshape(1, -1),
    title_vector.reshape(1, -1)
)[0, 0]

# Display the score.
print(f"Similarity score: {similarity_score:.4f}")

Similarity score: 0.7314


In [74]:
#Display the top candidates
# Choose the columns to display.
ranking_display_columns = [
    'word2vec_rank',
    'id',
    'job_title',
    'word2vec_similarity',
    'fit'
]

# Select the requested number of highest-ranked candidates.
top_word2vec_candidates = word2vec_ranking[
    ranking_display_columns
].head(DISPLAY_CANDIDATE_COUNT)

# Display the shortlist.
display(top_word2vec_candidates)

,word2vec_rank,id,job_title,word2vec_similarity,fit
0,1,73,"Aspiring Human Resources Manager, seeking inte...",0.900601,0.950300
1,2,74,Human Resources Professional,0.877760,0.938880
2,3,6,Aspiring Human Resources Specialist,0.872995,0.936497
3,4,3,Aspiring Human Resources Professional,0.871868,0.935934
4,5,28,Seeking Human Resources Opportunities,0.870136,0.935068
5,6,99,Seeking Human Resources Position,0.869522,0.934761
6,7,10,Seeking Human Resources HRIS and Generalist Po...,0.793605,0.896803
7,8,101,Human Resources Generalist at Loparex,0.783363,0.891682
8,9,88,Human Resources Management Major,0.777377,0.888688
9,10,100,Aspiring Human Resources Manager | Graduating ...,0.774377,0.887189


**Interpretation:**
- Inspect the current top results after duplicate removal.
- Each cleaned description appears once; ties use the representative ID.
- `fit` is a relevance score, not a probability of suitability.
- Human review is needed to confirm suitability.

## Load a Sentence-BERT-style model

Use a pretrained sentence embedding model instead of manually averaging Word2Vec word vectors. `all-MiniLM-L6-v2` is a compact Sentence Transformers model with a MiniLM backbone; it is not the original `bert-base-uncased` model. It is trained for sentence representations, making it a practical BERT-family comparison for semantic ranking.

The first run downloads model weights. CPU is sufficient for this candidate list. No fine-tuning is performed on the blank `fit` column. [Sentence Transformers usage](https://www.sbert.net/docs/sentence_transformer/usage/usage.html).


In [75]:
# Load the sentence model configured in your notebook.
bert_model = SentenceTransformer(SENTENCE_BERT_MODEL_NAME)

# Create one vector for each candidate's job title.
bert_vectors = bert_model.encode(
    prepared_candidate_data['job_title'].tolist(),
    normalize_embeddings=True
)

# Create a vector for the first search phrase.
phrase_vectors1 = bert_model.encode(
    ROLE_SEARCH_PHRASES[0],
    normalize_embeddings=True
)

# Create a vector for the second search phrase.
phrase_vectors2 = bert_model.encode(
    ROLE_SEARCH_PHRASES[1],
    normalize_embeddings=True
)
print('Sentence-BERT candidate matrix:', candidate_bert_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence-BERT candidate matrix: (52, 384)


**Interpretation:** `all-MiniLM-L6-v2` produces one 384-dimensional vector per unique description. The printed matrix shape gives the actual row count. Compare each candidate only with a query encoded by the same model.

In [76]:
#Combine the two search vectors
# Average both search phrases into one role vector.
bert_role_vector = (phrase_vectors1 + phrase_vectors2) / 2

# Reshape into one row and normalize to length 1.
bert_role_vector = normalize(bert_role_vector.reshape(1, -1))

In [77]:
#Calculate candidate similarity scores
# Compare each candidate's title vector with the role vector.
bert_similarity_scores = cosine_similarity(
    bert_vectors,
    bert_role_vector
).ravel()

# Keep scores within the valid cosine range.
bert_similarity_scores = np.clip(bert_similarity_scores, -1, 1)

In [78]:
# Create the results table
# Copy candidate details into a separate table.
bert_ranking = prepared_candidate_data.copy()

# Add each candidate's similarity score.
bert_ranking['bert_similarity'] = bert_similarity_scores

# Convert similarity into a 0–1 relevance score, not a probability.
bert_ranking['fit'] = (bert_similarity_scores + 1) / 2

In [79]:
#Sort candidates and assign ranks
# Sort by highest similarity, then lowest ID for tied scores.
bert_ranking = bert_ranking.sort_values(
    ['bert_similarity', 'id'],
    ascending=[False, True]
).reset_index(drop=True)

# Add rank numbers starting from 1.
bert_ranking.insert(
    0,
    'bert_rank',
    np.arange(1, len(bert_ranking) + 1)
)

In [80]:
# Display the top candidates
# Select the columns to display.
bert_display_columns = [
    'bert_rank',
    'id',
    'job_title',
    'bert_similarity',
    'fit'
]

# Display the highest-ranked candidates.
display(
    bert_ranking[bert_display_columns].head(DISPLAY_CANDIDATE_COUNT)
)

,bert_rank,id,job_title,bert_similarity,fit
0,1,3,Aspiring Human Resources Professional,0.904441,0.952220
1,2,99,Seeking Human Resources Position,0.899401,0.949701
2,3,6,Aspiring Human Resources Specialist,0.897195,0.948597
3,4,28,Seeking Human Resources Opportunities,0.892000,0.946000
4,5,10,Seeking Human Resources HRIS and Generalist Po...,0.815279,0.907639
5,6,74,Human Resources Professional,0.799030,0.899515
6,7,67,"Human Resources, Staffing and Recruiting Profe...",0.774065,0.887033
7,8,100,Aspiring Human Resources Manager | Graduating ...,0.754122,0.877061
8,9,76,Aspiring Human Resources Professional | Passio...,0.741252,0.870626
9,10,82,Aspiring Human Resources Professional | An ene...,0.737555,0.868777


**Interpretation:**
- Inspect the current top descriptions and scores.
- Duplicate cleaned descriptions no longer occupy multiple ranking positions.
- A high title similarity requires human review; `fit` is not a probability.
- Rank changes alone do not establish improved quality.

## 15. Compare all three rankings

Compare positions, not raw similarity magnitudes across different representations. A changed order does not establish improvement; independent relevance labels are needed.


In [81]:
# Start with the ID and job title for each unique description.
ranking_comparison = prepared_candidate_data[['id','job_title']].merge(
     # Select the TF–IDF rank and similarity score for each ID matching records using the candidate Id. IF each ID appear at most once in both tables
    tfidf_ranking[['id','tfidf_rank','tfidf_similarity']], on='id', validate='one_to_one'
).merge( # Add the Word2Vec results to the combined table.
    word2vec_ranking[['id','word2vec_rank','word2vec_similarity']], on='id', validate='one_to_one'
).merge( # Add the BERT results to the combined table.
    bert_ranking[['id','bert_rank','bert_similarity']], on='id', validate='one_to_one'
)
# Sort by BERT rank and display the requested number of results.
display(ranking_comparison.sort_values('bert_rank').head(DISPLAY_CANDIDATE_COUNT))

# Show which original candidate IDs each ranked description represents.
display(ranking_comparison.merge(
    prepared_candidate_data[['id', 'associated_candidate_ids', 'description_record_count']],
    on='id', validate='one_to_one'
).sort_values('bert_rank').head(DISPLAY_CANDIDATE_COUNT))

# Verify all ranking methods contain every unique description exactly once.
for ranking_table in [tfidf_ranking, word2vec_ranking, bert_ranking]:
    assert ranking_table['id'].is_unique
    assert set(ranking_table['id']) == set(prepared_candidate_data['id'])
print('Passed: all three rankings contain one result per cleaned description.')

,id,job_title,tfidf_rank,tfidf_similarity,word2vec_rank,word2vec_similarity,bert_rank,bert_similarity
2,3,Aspiring Human Resources Professional,1,0.586905,4,0.871868,1,0.904441
46,99,Seeking Human Resources Position,3,0.544027,6,0.869522,2,0.899401
5,6,Aspiring Human Resources Specialist,4,0.514304,3,0.872995,3,0.897195
13,28,Seeking Human Resources Opportunities,2,0.544027,5,0.870136,4,0.892000
8,10,Seeking Human Resources HRIS and Generalist Po...,8,0.318087,7,0.793605,5,0.815279
22,74,Human Resources Professional,6,0.364647,2,0.877760,6,0.799030
15,67,"Human Resources, Staffing and Recruiting Profe...",26,0.153296,16,0.732814,7,0.774065
47,100,Aspiring Human Resources Manager | Graduating ...,12,0.264439,10,0.774377,8,0.754122
24,76,Aspiring Human Resources Professional | Passio...,24,0.167974,24,0.678140,9,0.741252
30,82,Aspiring Human Resources Professional | An ene...,16,0.224944,23,0.682585,10,0.737555


,id,job_title,tfidf_rank,tfidf_similarity,word2vec_rank,word2vec_similarity,bert_rank,bert_similarity,associated_candidate_ids,description_record_count
2,3,Aspiring Human Resources Professional,1,0.586905,4,0.871868,1,0.904441,"[3, 17, 21, 33, 46, 58, 97]",7
46,99,Seeking Human Resources Position,3,0.544027,6,0.869522,2,0.899401,[99],1
5,6,Aspiring Human Resources Specialist,4,0.514304,3,0.872995,3,0.897195,"[6, 24, 36, 49, 60]",5
13,28,Seeking Human Resources Opportunities,2,0.544027,5,0.870136,4,0.892000,"[28, 30]",2
8,10,Seeking Human Resources HRIS and Generalist Po...,8,0.318087,7,0.793605,5,0.815279,"[10, 40, 53, 62]",4
22,74,Human Resources Professional,6,0.364647,2,0.877760,6,0.799030,[74],1
15,67,"Human Resources, Staffing and Recruiting Profe...",26,0.153296,16,0.732814,7,0.774065,[67],1
47,100,Aspiring Human Resources Manager | Graduating ...,12,0.264439,10,0.774377,8,0.754122,[100],1
24,76,Aspiring Human Resources Professional | Passio...,24,0.167974,24,0.678140,9,0.741252,[76],1
30,82,Aspiring Human Resources Professional | An ene...,16,0.224944,23,0.682585,10,0.737555,[82],1


Passed: all three rankings contain one result per cleaned description.


## Learn a ranking from starred candidates

Use the top 10 Sentence-BERT descriptions as initial stars, or supply your own candidate IDs. Label starred descriptions `1` and the remaining descriptions `0` (not starred, not confirmed irrelevant).

Features are the three existing similarity scores: TF–IDF, Word2Vec, and Sentence-BERT. Candidate IDs, location, connections, and review flags are not scoring features. This is a single-role pairwise learning-to-rank model: learn to score a starred description above an unstarred description.

In [82]:
# Import the two tools needed for pairwise ranking.
from sklearn.preprocessing import StandardScaler

# Leave empty to star the top 10 Sentence-BERT results automatically.
# To use your own selections, enter their original or representative IDs here.
MANUALLY_STARRED_CANDIDATE_IDS = []
NUMBER_OF_INITIAL_STARS = 10

# Use only the similarity features already calculated .
RANKING_FEATURE_COLUMNS = [
    'tfidf_similarity',
    'word2vec_similarity',
    'bert_similarity'
]


### Assemble one feature row per unique description

In [83]:
# Start with the unique cleaned descriptions and their representative IDs.
ranking_training_data = prepared_candidate_data[
    ['id', 'job_title', 'clean_job_title', 'associated_candidate_ids']
].copy()

# Loop through each ranking table and its similarity column.
for model_ranking, score_column in [
    (tfidf_ranking, 'tfidf_similarity'),
    (word2vec_ranking, 'word2vec_similarity'),
    (bert_ranking, 'bert_similarity')
]:  # Add this model's similarity scores to the training table.
    ranking_training_data = ranking_training_data.merge(
        model_ranking[['id', score_column]],
        on='id', how='left', validate='one_to_one'
    )

# Confirm that every description has finite scores in all three methods.
if not np.isfinite(ranking_training_data[RANKING_FEATURE_COLUMNS].to_numpy()).all():
    raise ValueError('Missing or invalid scores: rerun all three ranking methods.')
assert ranking_training_data['clean_job_title'].is_unique

display(ranking_training_data.head(10))


,id,job_title,clean_job_title,associated_candidate_ids,tfidf_similarity,word2vec_similarity,bert_similarity
0,1,2019 C.T. Bauer College of Business Graduate (...,2019 c t bauer college of business graduate ma...,"[1, 14, 15, 19, 31, 44, 57]",0.179593,0.490347,0.525391
1,2,Native English Teacher at EPIK (English Progra...,native english teacher at epik english program...,"[2, 16, 20, 32, 45]",0.000000,0.193101,0.243794
2,3,Aspiring Human Resources Professional,aspiring human resources professional,"[3, 17, 21, 33, 46, 58, 97]",0.586905,0.871868,0.904441
3,4,People Development Coordinator at Ryan,people development coordinator at ryan,"[4, 18, 22, 34, 47, 59]",0.000000,0.317379,0.397721
4,5,Advisory Board Member at Celal Bayar University,advisory board member at celal bayar university,"[5, 23, 35, 48]",0.000000,0.244476,0.242718
5,6,Aspiring Human Resources Specialist,aspiring human resources specialist,"[6, 24, 36, 49, 60]",0.514304,0.872995,0.897195
6,7,Student at Humber College and Aspiring Human R...,student at humber college and aspiring human r...,"[7, 9, 25, 37, 39, 50, 52]",0.263188,0.741314,0.728456
7,8,HR Senior Specialist,human resources senior specialist,"[8, 26, 38, 51, 61]",0.207117,0.732152,0.512262
8,10,Seeking Human Resources HRIS and Generalist Po...,seeking human resources hris and generalist po...,"[10, 40, 53, 62]",0.318087,0.793605,0.815279
9,11,Student at Chapman University,student at chapman university,"[11, 41, 54, 63]",0.000000,0.218949,0.331598


### Mark starred descriptions as 1 and the others as 0

In [84]:
# Map original candidate IDs to the representative of their unique description.
original_to_representative = candidate_description_mapping.set_index(
    'id'                          # Use original candidate IDs as dictionary keys.
)['representative_id'].to_dict()  # Use representative IDs as dictionary values.

if MANUALLY_STARRED_CANDIDATE_IDS:
    # Reject unknown IDs rather than silently ignoring a selection.
    unknown_ids = set(MANUALLY_STARRED_CANDIDATE_IDS) - set(original_to_representative)
    if unknown_ids:
        raise ValueError(f'Unknown selected candidate IDs: {sorted(unknown_ids)}')
 
    # Multiple selected IDs sharing a title count as one starred description.
    starred_candidate_ids = sorted({
        original_to_representative[candidate_id]
        for candidate_id in MANUALLY_STARRED_CANDIDATE_IDS
    })
    label_source = 'manual candidate selections'
else:
    # Keep at least one unstarred description so preference pairs can be formed.
    if len(ranking_training_data) <= NUMBER_OF_INITIAL_STARS:
        raise ValueError('Need more than 10 unique descriptions, or select fewer stars manually.')

    # Use the duplicate-free Sentence-BERT ranking for the initial top 10.
    starred_candidate_ids = bert_ranking.sort_values(
        ['bert_similarity', 'id'], ascending=[False, True]
    ).head(NUMBER_OF_INITIAL_STARS)['id'].tolist()
    label_source = 'automatic Sentence-BERT top 10 (provisional labels)'

# 1 means starred; 0 means not starred.
ranking_training_data['starred'] = ranking_training_data['id'].isin(
    starred_candidate_ids
).astype(int)

# Both groups are required for pairwise training.
if ranking_training_data['starred'].nunique() != 2:
    raise ValueError('Select at least one starred and one unstarred description.')

print('Label source:', label_source)
print('Starred representative IDs:', starred_candidate_ids)
display(ranking_training_data['starred'].value_counts().rename_axis('starred').to_frame('descriptions'))
display(ranking_training_data.sort_values(
    ['starred', 'bert_similarity', 'id'], ascending=[False, False, True]
))


Label source: automatic Sentence-BERT top 10 (provisional labels)
Starred representative IDs: [3, 99, 6, 28, 10, 74, 67, 100, 76, 82]


,descriptions
starred,
0,42
1,10


,id,job_title,clean_job_title,associated_candidate_ids,tfidf_similarity,word2vec_similarity,bert_similarity,starred
2,3,Aspiring Human Resources Professional,aspiring human resources professional,"[3, 17, 21, 33, 46, 58, 97]",0.586905,0.871868,0.904441,1
46,99,Seeking Human Resources Position,seeking human resources position,[99],0.544027,0.869522,0.899401,1
5,6,Aspiring Human Resources Specialist,aspiring human resources specialist,"[6, 24, 36, 49, 60]",0.514304,0.872995,0.897195,1
13,28,Seeking Human Resources Opportunities,seeking human resources opportunities,"[28, 30]",0.544027,0.870136,0.892000,1
8,10,Seeking Human Resources HRIS and Generalist Po...,seeking human resources hris and generalist po...,"[10, 40, 53, 62]",0.318087,0.793605,0.815279,1
22,74,Human Resources Professional,human resources professional,[74],0.364647,0.877760,0.799030,1
15,67,"Human Resources, Staffing and Recruiting Profe...",human resources staffing and recruiting profes...,[67],0.153296,0.732814,0.774065,1
47,100,Aspiring Human Resources Manager | Graduating ...,aspiring human resources manager graduating ma...,[100],0.264439,0.774377,0.754122,1
24,76,Aspiring Human Resources Professional | Passio...,aspiring human resources professional passiona...,[76],0.167974,0.678140,0.741252,1
30,82,Aspiring Human Resources Professional | An ene...,aspiring human resources professional an energ...,[82],0.224944,0.682585,0.737555,1


In [85]:
# Keep the training table and starred labels created in the preceding cells.
# Rebuilding ranking_training_data here would remove its starred column.
if 'starred' not in ranking_training_data.columns:
    raise ValueError('Run the starred-label cell above before training the ranker.')

# Confirm that every description still has a valid label and complete scores.
if not ranking_training_data['starred'].isin([0, 1]).all():
    raise ValueError('Starred labels must be 0 or 1.')
if ranking_training_data['starred'].nunique() != 2:
    raise ValueError('Select at least one starred and one unstarred description.')
if not np.isfinite(ranking_training_data[RANKING_FEATURE_COLUMNS].to_numpy()).all():
    raise ValueError('Missing or invalid scores: rerun all three ranking methods.')
assert ranking_training_data['clean_job_title'].is_unique

display(ranking_training_data.head(10))


,id,job_title,clean_job_title,associated_candidate_ids,tfidf_similarity,word2vec_similarity,bert_similarity,starred
0,1,2019 C.T. Bauer College of Business Graduate (...,2019 c t bauer college of business graduate ma...,"[1, 14, 15, 19, 31, 44, 57]",0.179593,0.490347,0.525391,0
1,2,Native English Teacher at EPIK (English Progra...,native english teacher at epik english program...,"[2, 16, 20, 32, 45]",0.000000,0.193101,0.243794,0
2,3,Aspiring Human Resources Professional,aspiring human resources professional,"[3, 17, 21, 33, 46, 58, 97]",0.586905,0.871868,0.904441,1
3,4,People Development Coordinator at Ryan,people development coordinator at ryan,"[4, 18, 22, 34, 47, 59]",0.000000,0.317379,0.397721,0
4,5,Advisory Board Member at Celal Bayar University,advisory board member at celal bayar university,"[5, 23, 35, 48]",0.000000,0.244476,0.242718,0
5,6,Aspiring Human Resources Specialist,aspiring human resources specialist,"[6, 24, 36, 49, 60]",0.514304,0.872995,0.897195,1
6,7,Student at Humber College and Aspiring Human R...,student at humber college and aspiring human r...,"[7, 9, 25, 37, 39, 50, 52]",0.263188,0.741314,0.728456,0
7,8,HR Senior Specialist,human resources senior specialist,"[8, 26, 38, 51, 61]",0.207117,0.732152,0.512262,0
8,10,Seeking Human Resources HRIS and Generalist Po...,seeking human resources hris and generalist po...,"[10, 40, 53, 62]",0.318087,0.793605,0.815279,1
9,11,Student at Chapman University,student at chapman university,"[11, 41, 54, 63]",0.000000,0.218949,0.331598,0


### Train the LightGBM ranker
LightGBM uses the three similarity scores and the starred labels directly. All descriptions compete for the same role, so they form one query group. The LambdaRank objective learns a tree-based score that favors starred descriptions over unstarred descriptions. There is no need to construct feature-difference pairs or scale the input features manually. The predicted score is not a suitability probability.

In [86]:
# Run once if LightGBM is not installed.
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [87]:
# Import the LightGBM ranking model.
from lightgbm import LGBMRanker

## Prepare features and labels

In [88]:
# Select the similarity scores used as model inputs.
RANKING_FEATURE_COLUMNS = [
    'tfidf_similarity',
    'word2vec_similarity',
    'bert_similarity'
]

# Keep the features in the same row order as the candidate table.
ranking_features = ranking_training_data[RANKING_FEATURE_COLUMNS].copy()

# Use your existing starred labels: 1 = selected, 0 = not selected.
ranking_labels = ranking_training_data['starred'].copy()

# Stop if any feature contains missing or infinite values.
if not np.isfinite(ranking_features.to_numpy()).all():
    raise ValueError('Ranking features contain missing or invalid values.')

# Require both starred and unstarred descriptions.
if set(ranking_labels.unique()) != {0, 1}:
    raise ValueError('Labels must contain both 0 and 1.')

# All descriptions compete for one role, so they form ONE ranking group.
ranking_group_sizes = [len(ranking_training_data)]

## Create and train the model

In [89]:
# Use small trees as a starting point for this small dataset.
learning_to_rank_model = LGBMRanker(
    objective='lambdarank',  # Learn to place higher-label candidates first.
    n_estimators=100,       # Maximum number of boosting trees.
    learning_rate=0.05,     # Control how strongly each tree updates scores.
    num_leaves=7,           # Limit tree complexity.
    max_depth=3,            # Limit tree depth.
    min_child_samples=3,    # Allow splits in this small dataset.
    reg_lambda=1.0,         # Add regularization.
    importance_type='gain',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Fit using candidate features, starred labels, and the role group.
learning_to_rank_model.fit(
    ranking_features,
    ranking_labels,
    group=ranking_group_sizes
)

,boosting_type,'gbdt'
,num_leaves,7
,max_depth,3
,learning_rate,0.05
,n_estimators,100
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,3


## Score and rank candidates

In [90]:
# Copy the candidate information.
learning_to_rank_results = ranking_training_data.copy()

# LightGBM uses predict(), rather than decision_function().
learning_to_rank_results['ltr_score'] = learning_to_rank_model.predict(
    ranking_features
)

# Sort by highest learned score; use ID to break ties.
learning_to_rank_results = learning_to_rank_results.sort_values(
    ['ltr_score', 'id'],
    ascending=[False, True]
).reset_index(drop=True)

# Add rank numbers starting from 1.
learning_to_rank_results.insert(
    0,
    'ltr_rank',
    np.arange(1, len(learning_to_rank_results) + 1)
)

# Add the original BERT rank for comparison.
learning_to_rank_results = learning_to_rank_results.merge(
    bert_ranking[['id', 'bert_rank']],
    on='id',
    how='left',
    validate='one_to_one'
)

# Display the new top 10.
display(
    learning_to_rank_results[
        ['ltr_rank', 'bert_rank', 'id', 'job_title', 'starred', 'ltr_score']
    ].head(10)
)

,ltr_rank,bert_rank,id,job_title,starred,ltr_score
0,1,1,3,Aspiring Human Resources Professional,1,1.964656
1,2,3,6,Aspiring Human Resources Specialist,1,1.964656
2,3,5,10,Seeking Human Resources HRIS and Generalist Po...,1,1.964656
3,4,4,28,Seeking Human Resources Opportunities,1,1.964656
4,5,7,67,"Human Resources, Staffing and Recruiting Profe...",1,1.964656
5,6,6,74,Human Resources Professional,1,1.964656
6,7,9,76,Aspiring Human Resources Professional | Passio...,1,1.964656
7,8,10,82,Aspiring Human Resources Professional | An ene...,1,1.964656
8,9,2,99,Seeking Human Resources Position,1,1.964656
9,10,8,100,Aspiring Human Resources Manager | Graduating ...,1,1.964656


## Display feature importance

In [91]:
# Tree models provide feature importance rather than linear coefficients.
ranking_feature_importance = pd.DataFrame({
    'feature': RANKING_FEATURE_COLUMNS,
    'gain_importance': learning_to_rank_model.feature_importances_
})

# Show features contributing most to training split improvements.
display(
    ranking_feature_importance.sort_values(
        'gain_importance',
        ascending=False
    ).reset_index(drop=True)
)

,feature,gain_importance
0,bert_similarity,98.093788
1,tfidf_similarity,0.000000
2,word2vec_similarity,0.000000


Interpretation:
- starred = 1 records your initial preference; it is not a prediction. 0 means not starred.
- ltr_rank is the new order; bert_rank is the previous order.
- Higher ltr_score means a stronger learned preference. It is not a probability.
- The tree-based model learns from the three similarity features; it does not strictly pin every starred description to the top 10.
- Review the results and replace provisional top-10 labels with actual judgments before claiming improvement.
- To evaluate independently, reserve description groups first, use only feedback groups for starring and training, choose settings on calibration groups, and evaluate the fixed model on untouched test groups. No training-set accuracy or NDCG is presented as evidence of quality here.

Conclusions and limits
- TF–IDF provides an explainable word-matching baseline; Word2Vec adds pretrained word representations; Sentence-BERT adds contextual title representations.
- Location and connections do not contribute to relevance scores. Unstarred candidates are not treated as irrelevant.
- Advertisement-like titles, sparse titles, and titles without known words remain reviewable.
- No automatic rejection cutoff or measured quality gain is claimed. Collect independent graded relevance judgments before comparing Precision@K, Recall@K, or NDCG@K.
- Rankings contain one representative per cleaned description, with original IDs preserved in the mapping.
- Keep identical normalized titles together when dividing feedback, validation, and test examples; do not evaluate feedback improvement on the same starred candidates.
- This notebook ranks title relevance and does not establish qualifications, talent, or job performance.

- **Ranking improvement with stars** : The model combines TF–IDF, Word2Vec, and BERT scores. Each star adds feedback, allowing it to learn our preferences and re-rank candidates. Check improvement using independently reviewed candidates; every star does not guarantee better results.
- **To filter unsuitable entries** : Remove confirmed advertisements and candidates who fail verified essential requirements. Send unclear profiles for review. Low similarity alone should not mean rejection.
- **One cutoff for every role**:No single cutoff can reliably fit every role or guarantee no missed talent. Choose and validate thresholds for each role, and audit low-ranked candidates.
- **Reducing human bias**: Use clear job-related criteria, multiple reviewers, hidden unrelated personal details, and random checks of low-ranked candidates. Automation can support consistency, but it cannot eliminate bias.